In [1]:
# Importing libraries!

In [ ]:
import pandas as pd
import json, os
import numpy as np

In [3]:
# Loading the Dataset

In [4]:
df = pd.read_parquet("data.parquet")
df.head()

,item_id,dept_id,cat_id,store_id,state_id,sales,date,wm_yr_wk,weekday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap,sell_price
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,2011,None,None,None,None,0,NaN
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,2011,None,None,None,None,0,NaN
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,2011,None,None,None,None,0,NaN
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,2011,None,None,None,None,0,NaN
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,2011,None,None,None,None,0,NaN


In [5]:
##########################################################################################################################################################################################

In [ ]:
# Drop unnecessary column

In [7]:
df.drop(columns=["wm_yr_wk"], inplace=True)
df.shape

(58327370, 16)

In [8]:
##########################################################################################################################################################################################

In [ ]:
# KPIs

In [10]:
df['revenue'] = df['sales'] * df['sell_price']

kpis = {
    "total_sales": int(df['sales'].sum()),
    "total_revenue": round(float(df['revenue'].sum()), 2),
    "date_range": {"start": str(df['date'].min().date()), "end": str(df['date'].max().date())},
    "unique_items": int(df['item_id'].nunique()),
    "unique_stores": int(df['store_id'].nunique()),
    "unique_states": int(df['state_id'].nunique()),
}

In [11]:
kpis

{'total_sales': 65695409,
 'total_revenue': 187676384.0,
 'date_range': {'start': '2011-01-29', 'end': '2016-04-24'},
 'unique_items': 3049,
 'unique_stores': 10,
 'unique_states': 3}

In [12]:
##########################################################################################################################################################################################

In [ ]:
# Time Series

In [14]:
daily = df.groupby('date')[['sales', 'revenue']].sum().reset_index()
daily['date'] = daily['date'].astype(str)
daily['revenue'] = daily['revenue'].round(2)

monthly = df.groupby(['year', 'month'])[['sales', 'revenue']].sum().reset_index()
monthly['revenue'] = monthly['revenue'].round(2)

yearly = df.groupby('year')[['sales', 'revenue']].sum().reset_index()
yearly['revenue'] = yearly['revenue'].round(2)

weekend_days = ['Saturday', 'Sunday']
df['is_weekend'] = df['weekday'].isin(weekend_days)
weekend = df.groupby('is_weekend')[['sales', 'revenue']].mean().reset_index()
weekend['revenue'] = weekend['revenue'].round(2)
weekend['is_weekend'] = weekend['is_weekend'].map({True: 'weekend', False: 'weekday'})

time_series = {
    "daily": daily.to_dict(orient='records'),
    "monthly": monthly.to_dict(orient='records'),
    "yearly": yearly.to_dict(orient='records'),
    "weekend_vs_weekday": weekend.to_dict(orient='records'),
}

In [15]:
daily.head()

,date,sales,revenue
0,2011-01-29,32631,81650.609375
1,2011-01-30,31749,78970.570312
2,2011-01-31,23783,57706.910156
3,2011-02-01,25412,60761.199219
4,2011-02-02,19146,46959.949219


In [16]:
##########################################################################################################################################################################################

In [ ]:
# Store Performance

In [ ]:
by_store = df.groupby('store_id')[['sales', 'revenue']].sum().reset_index()
by_store['revenue'] = by_store['revenue'].round(2)
kpis["total_revenue"] = round(float(by_store["revenue"].sum()), 2)

by_state = df.groupby('state_id')[['sales', 'revenue']].sum().reset_index()
by_state['revenue'] = by_state['revenue'].round(2)

by_store_monthly = df.groupby(['store_id', 'year', 'month'])[['sales', 'revenue']].sum().reset_index()
by_store_monthly['revenue'] = by_store_monthly['revenue'].round(2)

store_perf = {
    "by_store": by_store.to_dict(orient='records'),
    "by_state": by_state.to_dict(orient='records'),
    "by_store_monthly": by_store_monthly.to_dict(orient='records'),
}

In [19]:
by_store

,store_id,sales,revenue
0,CA_1,7698216,22520250.0
1,CA_2,5685475,17409444.0
2,CA_3,11188180,32108720.0
3,CA_4,4103676,12207148.0
4,TX_1,5595292,15729162.0
5,TX_2,7214384,20518238.0
6,TX_3,6089330,17796728.0
7,WI_1,5149062,14769235.0
8,WI_2,6544012,17690738.0
9,WI_3,6427782,16926906.0


In [20]:
##########################################################################################################################################################################################

In [ ]:
# Category & Department

In [22]:
by_cat = df.groupby('cat_id')[['sales', 'revenue']].sum().reset_index()
by_cat['revenue'] = by_cat['revenue'].round(2)

by_dept = df.groupby('dept_id')[['sales', 'revenue']].sum().reset_index()
by_dept['revenue'] = by_dept['revenue'].round(2)

by_cat_dept = df.groupby(['cat_id', 'dept_id'])[['sales', 'revenue']].sum().reset_index()
by_cat_dept['revenue'] = by_cat_dept['revenue'].round(2)

cat_dept = {
    "by_cat": by_cat.to_dict(orient='records'),
    "by_dept": by_dept.to_dict(orient='records'),
    "by_cat_dept": by_cat_dept.to_dict(orient='records'),
}

by_cat

,cat_id,sales,revenue
0,FOODS,45089939,108904624.0
1,HOBBIES,6124800,22818406.0
2,HOUSEHOLD,14480670,55953548.0


In [23]:
##########################################################################################################################################################################################

In [ ]:
# Event & SNAP

In [ ]:
snap = df.groupby('snap')[['sales', 'revenue']].mean().reset_index()
snap['revenue'] = snap['revenue'].round(2)
snap['snap'] = snap['snap'].map({1: 'snap', 0: 'no_snap'})

df['has_event'] = df['event_name_1'].notna() & (df['event_name_1'] != 'None')
event_impact = df.groupby('has_event')[['sales', 'revenue']].mean().reset_index()
event_impact['revenue'] = event_impact['revenue'].round(2)
event_impact['has_event'] = event_impact['has_event'].map({True: 'event', False: 'no_event'})

by_event_type = df[df['has_event']].groupby('event_type_1', observed=True)[['sales', 'revenue']].mean().reset_index()
by_event_type['revenue'] = by_event_type['revenue'].round(2)

top_events = df[df['has_event']].groupby('event_name_1', observed=True)[['sales', 'revenue']].mean().reset_index().sort_values('sales', ascending=False).head(20)
top_events['revenue'] = top_events['revenue'].round(2)

event_snap = {
    "snap_vs_no_snap": snap.to_dict(orient='records'),
    "event_vs_no_event": event_impact.to_dict(orient='records'),
    "by_event_type": by_event_type.to_dict(orient='records'),
    "top_events": top_events.to_dict(orient='records'),
}

event_impact

In [ ]:
# Products
top_items = df.groupby(['item_id', 'cat_id', 'dept_id'])[['sales', 'revenue']].sum().reset_index()
top_items['revenue'] = top_items['revenue'].round(2)
top_items = top_items.sort_values('sales', ascending=False).head(50)

top_items_by_store = df.groupby(['item_id', 'store_id'])[['sales', 'revenue']].sum().reset_index()
top_items_by_store['revenue'] = top_items_by_store['revenue'].round(2)
top_items_by_store = top_items_by_store.sort_values('sales', ascending=False).head(50)

products = {
    "top_items": top_items.to_dict(orient='records'),
    "top_items_by_store": top_items_by_store.to_dict(orient='records'),
}

top_items

In [30]:
##########################################################################################################################################################################################

In [ ]:
# Exporting
def round_floats(obj, decimals=2):
    if isinstance(obj, dict):
        return {k: round_floats(v, decimals) for k, v in obj.items()}
    if isinstance(obj, list):
        return [round_floats(x, decimals) for x in obj]
    if isinstance(obj, float):
        return round(obj, decimals)
    if hasattr(obj, "item"):
        v = obj.item()
        return round(v, decimals) if isinstance(v, float) else int(v) if isinstance(v, (np.integer, int)) else v
    return obj

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return round(float(obj), 2)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

OUT = "dashboard-data"
os.makedirs(OUT, exist_ok=True)

exports = {
    "kpis": kpis,
    "time-series": time_series,
    "store-perf": store_perf,
    "cat-dept": cat_dept,
    "event-snap": event_snap,
    "products": products,
}

for name, data in exports.items():
    data = round_floats(data)
    with open(f"{OUT}/{name}.json", "w") as f:
        json.dump(data, f, cls=NpEncoder)

In [32]:
##########################################################################################################################################################################################